In [2]:
import pandas as pd
import geopandas as gpd
from pathlib import Path 

In [3]:
project_root = Path.cwd()
if not Path('data').exists():
    project_root = project_root.parent 
print(project_root)

d:\Wu\2026\Project Portfolio\003 Project\grocery-demand-estimation


In [4]:
# data path
tract_path = project_root / 'data/raw/tract_2024/cb_2024_16_tract_500k.shp'
city_path = project_root / 'data/raw/place/cb_2025_16_place_500k.shp'
expenditure_path = project_root / 'data/processed/avg_expenditure_by_income.csv'
demography_path = project_root / 'data/processed/kootenai_demographics2024.csv'


In [5]:
# Read all datasets
tract = gpd.read_file(tract_path)
city = gpd.read_file(city_path)
expend = pd.read_csv(expenditure_path)
demo = pd.read_csv(demography_path)

In [6]:
print(tract.shape)
tract.head()

(456, 14)


,STATEFP,COUNTYFP,TRACTCE,GEOIDFQ,GEOID,NAME,NAMELSAD,STUSPS,NAMELSADCO,STATE_NAME,LSAD,ALAND,AWATER,geometry
0,16,017,950100,1400000US16017950100,16017950100,9501,Census Tract 9501,ID,Bonner County,Idaho,CT,1040606721,159680574,"POLYGON ((-116.50511 47.94953, -116.50481 47.9..."
1,16,059,970200,1400000US16059970200,16059970200,9702,Census Tract 9702,ID,Lemhi County,Idaho,CT,1230047586,1802039,"POLYGON ((-113.99543 45.41691, -113.9945 45.42..."
2,16,083,000300,1400000US16083000300,16083000300,3,Census Tract 3,ID,Twin Falls County,Idaho,CT,381182424,2847873,"POLYGON ((-114.95143 42.53916, -114.9495 42.54..."
3,16,039,960500,1400000US16039960500,16039960500,9605,Census Tract 9605,ID,Elmore County,Idaho,CT,25833164,92328,"POLYGON ((-115.89592 43.07025, -115.88581 43.0..."
4,16,001,010501,1400000US16001010501,16001010501,105.01,Census Tract 105.01,ID,Ada County,Idaho,CT,1022415884,5451232,"POLYGON ((-116.51362 43.38654, -116.51353 43.4..."


In [7]:
print(city.shape)
city.head()

(236, 13)


,STATEFP,PLACEFP,PLACENS,GEOIDFQ,GEOID,NAME,NAMELSAD,STUSPS,STATE_NAME,LSAD,ALAND,AWATER,geometry
0,16,34930,02410697,1600000US1634930,1634930,Hansen,Hansen city,ID,Idaho,25,903899,0,"POLYGON ((-114.30594 42.53688, -114.29864 42.5..."
1,16,39070,02410802,1600000US1639070,1639070,Huetter,Huetter city,ID,Idaho,25,117471,0,"POLYGON ((-116.85411 47.70502, -116.85161 47.7..."
2,16,27460,02410495,1600000US1627460,1627460,Ferdinand,Ferdinand city,ID,Idaho,25,452353,0,"POLYGON ((-116.39491 46.15449, -116.38965 46.1..."
3,16,67870,02410949,1600000US1667870,1667870,Riggins,Riggins city,ID,Idaho,25,1029902,123855,"POLYGON ((-116.32982 45.4122, -116.32138 45.41..."
4,16,16750,02410187,1600000US1616750,1616750,Coeur d'Alene,Coeur d'Alene city,ID,Idaho,25,43691994,1976736,"POLYGON ((-116.85288 47.70099, -116.85096 47.7..."


In [8]:
print(expend.shape)
expend.head()

(6, 9)


,Unnamed: 0,num_cu,pct_cu,income_mean,grocery,grocery_pct,grocery_group,avg_grocery,highest_income
0,Aggregate,135760.0,100.0,104207.0,842257.0,100.0,842257.000,6204.014437,9999999
1,lowest20pct,27139.0,20.0,16658.0,12.3,12.3,103597.611,3817.296547,29932
2,second20pct,27186.0,20.0,42925.0,15.9,15.9,133918.863,4926.023063,57452
3,third20pct,26959.0,19.9,74474.0,18.7,18.7,157502.059,5842.281205,94511
4,fourth20pct,27205.0,20.0,121548.0,23.0,23.0,193719.110,7120.717148,155925


In [9]:
print(demo.shape)
demo.head()

(39, 10)


,GEOID,state,county,tract,household_count,avg_household_size,median_income,median_age,owner_rate,college_plus_pct
0,16055000101,16,55,101,1202,2.59,97381,51.3,88.019967,35.781383
1,16055000102,16,55,102,1847,2.56,96493,49.5,89.767190,23.312883
2,16055000201,16,55,201,1330,2.92,109726,37.8,80.075188,26.440922
3,16055000202,16,55,202,1312,2.70,92500,53.7,96.493902,30.707457
4,16055000203,16,55,203,1407,2.41,70843,52.6,80.739161,26.635146


### Filter tract GeoDataFrame to Kootenai county

In [10]:
tract = tract[tract['COUNTYFP'] == '055'].copy()

In [12]:
print(tract.shape)
tract['COUNTYFP'].unique()

(39, 14)


<StringArray>
['055']
Length: 1, dtype: str

### Spatially filter city GeoDataFrame using Kootenai county boundary

In [13]:
# Double check both layers use the same crs
print(tract.crs)
print(city.crs)

EPSG:4269
EPSG:4269


In [14]:
city = gpd.clip(city, tract)

In [16]:
print(city.shape)
print(city['NAME'])

(16, 13)
38           Conkling Park
47                  Worley
217               Harrison
162           Rockford Bay
22     Fernan Lake Village
4            Coeur d'Alene
1                  Huetter
208             State Line
48              Post Falls
182         Dalton Gardens
105                 Hauser
192            Hayden Lake
86                  Hayden
185               Rathdrum
88                   Athol
130            Spirit Lake
Name: NAME, dtype: str


### Join demographic table to tract GeoDataFrame

In [21]:
print(tract['GEOID'].dtype)
tract['GEOID'].head()

str


18    16055001901
19    16055000902
37    16055001300
69    16055000703
70    16055000800
Name: GEOID, dtype: str

In [ ]:
print(demo['GEOID'].dtype)   # demo.dtypes 
demo['GEOID'].head()

int64


0    16055000101
1    16055000102
2    16055000201
3    16055000202
4    16055000203
Name: GEOID, dtype: int64

In [26]:
demo["GEOID"] = demo["GEOID"].astype(str)
demo['GEOID'].dtype

<StringDtype(storage='python', na_value=nan)>

In [27]:
tract = tract.merge(demo, on='GEOID', how='left')

In [28]:
print(tract.shape)
tract.head()

(39, 23)


,STATEFP,COUNTYFP,TRACTCE,GEOIDFQ,GEOID,NAME,NAMELSAD,STUSPS,NAMELSADCO,STATE_NAME,...,geometry,state,county,tract,household_count,avg_household_size,median_income,median_age,owner_rate,college_plus_pct
0,16,055,001901,1400000US16055001901,16055001901,19.01,Census Tract 19.01,ID,Kootenai County,Idaho,...,"POLYGON ((-116.7505 47.45958, -116.74964 47.46...",16,55,1901,825,2.47,132678,60.6,98.424242,35.437761
1,16,055,000902,1400000US16055000902,16055000902,9.02,Census Tract 9.02,ID,Kootenai County,Idaho,...,"POLYGON ((-116.81765 47.69942, -116.81537 47.7...",16,55,902,1715,1.62,42361,52.8,25.830904,21.954023
2,16,055,001300,1400000US16055001300,16055001300,13,Census Tract 13,ID,Kootenai County,Idaho,...,"POLYGON ((-116.78629 47.69826, -116.78626 47.7...",16,55,1300,1857,2.25,73051,33.3,49.703823,17.790201
3,16,055,000703,1400000US16055000703,16055000703,7.03,Census Tract 7.03,ID,Kootenai County,Idaho,...,"POLYGON ((-116.82978 47.7283, -116.82978 47.73...",16,55,703,3411,2.37,81778,36.9,78.803870,35.961398
4,16,055,000800,1400000US16055000800,16055000800,8,Census Tract 8,ID,Kootenai County,Idaho,...,"POLYGON ((-116.85126 47.7194, -116.85123 47.72...",16,55,800,2652,2.27,79440,47.5,66.855204,31.303419


### Assign average grocery expenditure

In [30]:
expend.head()

,Unnamed: 0,num_cu,pct_cu,income_mean,grocery,grocery_pct,grocery_group,avg_grocery,highest_income
0,Aggregate,135760.0,100.0,104207.0,842257.0,100.0,842257.000,6204.014437,9999999
1,lowest20pct,27139.0,20.0,16658.0,12.3,12.3,103597.611,3817.296547,29932
2,second20pct,27186.0,20.0,42925.0,15.9,15.9,133918.863,4926.023063,57452
3,third20pct,26959.0,19.9,74474.0,18.7,18.7,157502.059,5842.281205,94511
4,fourth20pct,27205.0,20.0,121548.0,23.0,23.0,193719.110,7120.717148,155925


In [32]:
ce = expend[1:]

In [33]:
ce.head()

,Unnamed: 0,num_cu,pct_cu,income_mean,grocery,grocery_pct,grocery_group,avg_grocery,highest_income
1,lowest20pct,27139.0,20.0,16658.0,12.3,12.3,103597.611,3817.296547,29932
2,second20pct,27186.0,20.0,42925.0,15.9,15.9,133918.863,4926.023063,57452
3,third20pct,26959.0,19.9,74474.0,18.7,18.7,157502.059,5842.281205,94511
4,fourth20pct,27205.0,20.0,121548.0,23.0,23.0,193719.110,7120.717148,155925
5,highest20pct,27272.0,20.1,264510.0,30.0,30.0,252677.100,9265.074069,9999999


### Write lookup function 
### to get average grocery expenditure per household based on income brakets

In [34]:
"""
Return the average annual grocery expenditure
based on the Consumer Expenditure income brackets.
"""
def get_avg_grocery(median_income):
    for i, row in ce.iterrows():
        if median_income <= row['highest_income']:
            return row['avg_grocery']
    return None 

In [35]:
print(get_avg_grocery(25000))
print(get_avg_grocery(60000))
print(get_avg_grocery(82000))
print(get_avg_grocery(180000))

3817.296547404105
5842.281204792462
5842.281204792462
9265.07406864183


In [36]:
tract['avg_grocery'] = tract['median_income'].apply(get_avg_grocery)

### Calculate potential grocery demand:
#### Estimated annual grocery spending generated by households living in that census tract

In [38]:
tract['potential_grocery_deman'] = (
    tract['household_count'] * tract['avg_grocery']
)

In [41]:
tract.columns

Index(['STATEFP', 'COUNTYFP', 'TRACTCE', 'GEOIDFQ', 'GEOID', 'NAME',
       'NAMELSAD', 'STUSPS', 'NAMELSADCO', 'STATE_NAME', 'LSAD', 'ALAND',
       'AWATER', 'geometry', 'state', 'county', 'tract', 'household_count',
       'avg_household_size', 'median_income', 'median_age', 'owner_rate',
       'college_plus_pct', 'avg_grocery', 'potential_grocery_deman'],
      dtype='str')

In [42]:
tract.isna().sum()

STATEFP                    0
COUNTYFP                   0
TRACTCE                    0
GEOIDFQ                    0
GEOID                      0
NAME                       0
NAMELSAD                   0
STUSPS                     0
NAMELSADCO                 0
STATE_NAME                 0
LSAD                       0
ALAND                      0
AWATER                     0
geometry                   0
state                      0
county                     0
tract                      0
household_count            0
avg_household_size         0
median_income              0
median_age                 0
owner_rate                 0
college_plus_pct           0
avg_grocery                0
potential_grocery_deman    0
dtype: int64

In [43]:
tract['potential_grocery_deman'].describe()

count    3.900000e+01
mean     1.116910e+07
std      4.648388e+06
min      3.324258e+06
25%      7.641704e+06
50%      1.070306e+07
75%      1.382728e+07
max      2.245773e+07
Name: potential_grocery_deman, dtype: float64

### Save data

In [44]:
output_path = project_root / 'data/processed/potential_grocery_demand.gpkg'

In [45]:
tract.to_file(
    output_path, layer='tract_demand', driver='GPKG'
)

In [46]:
city_outpath = project_root / 'data/processed/kootenai_cities.gpkg'
city.to_file(
    city_outpath, layer='cities', driver='GPKG'
)